In [ ]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# Fair-comparison transforms: every TEXT baseline reads the SAME Wikidata
# label-expanded query text SparqlCE reads (P31 -> "instance of (P31)").
# SparqlCE expands internally via its graph builder, so it takes NO transform.
from sparqlce.sparql_utils import expand_wikidata_labels as WDX
E5_XFORM = lambda q: f"query: {WDX(q)}"   # E5 family: its "query: " prefix + expand
TXT_XFORM = WDX                            # other encoders / lexical: expand only

# 2. Define File Paths
# Point this directly to your test file
TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "wikidata_testset_RQ1.json") 
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "wikidata", "best_model.pt")

TOPKS = [3, 5, 10, 20]
BATCH = 32

# 3. Load the test dataset directly
test_data = load_triplets(TEST_DATA_PATH)

print(f"Dataset Loaded -> Test Split: {len(test_data)}")

# 4. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")
# Enable per-bin lexical stratification for every model below:
#   [0,0.3) adversarial | [0.3,0.5) divergent | [0.5,0.8) moderate | [0.8,1.0) near-identical
import evaluate_rq1
evaluate_rq1.STRATIFY_BINS = [0.3, 0.5, 0.8]  # cumulative: cos < threshold


In [ ]:
# SPARQLCE — no transform: it expands P/Q ids internally via its graph builder.
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF (fit on label-expanded corpus + anchor)
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)

# BM25 (tokenize label-expanded corpus + anchor)
evaluate_bm25(eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)

# E5 (zero-shot) — "query: " prefix + label expansion
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=E5_XFORM)

# Qwen2.5-Coder (generative code LLM used as an embedder, last-token pooling;
# 0.5B ≈ scale-matched to the dense 0.6B Qwen3-Embedding it replaces)
evaluate_code_llm('Qwen/Qwen2.5-Coder-0.5B', 'Qwen2.5-Coder-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)

In [ ]:
# Prior-literature SPARQL / code encoder baselines (zero-shot, mean-pooled):
#   CodeBERT (SparqlCE's own backbone), GraphCodeBERT (CodeBERT + data-flow
#   graph), SPBERT (BERT pretrained on SPARQL). Defined in evaluate_rq1.py as
#   SPARQL_ENCODER_BASELINES. The "mean pooling" notice each emits is correct
#   here (raw MLM encoders), not the LoRA-adapter bug.
# transform=TXT_XFORM: same label-expanded input as every other text baseline.
evaluate_sparql_encoder_baselines(eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH, transform=TXT_XFORM)

In [ ]:
display_name = "SBERT/MiniLM" 
model_id = "sentence-transformers/all-MiniLM-L6-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)


In [ ]:
display_name = "SBERT/MPNet"
model_id = "sentence-transformers/all-mpnet-base-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=TXT_XFORM)


In [ ]:
# E5-base-v2 (fine-tuned on SQuARE-Wikidata) — capacity-matched trained baseline.
# Checkpoint saved by 05_e5_contrastive_finetune_square_wikidata.ipynb (which now
# also trains on label-expanded text). Eval transform = "query: " + label expansion,
# matching that notebook's own Section 10 encoding.
E5_FT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "e5_square_wikidata", "best_model")
evaluate_sentence_transformer(
    E5_FT_PATH,
    "E5-base-v2 (fine-tuned)",
    eval_items,
    global_corpus,
    relevance_scores_per_item,
    TOPKS,
    transform=E5_XFORM,
)